# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is structured according to the [Croissant schema](https://mlcommons.org/croissant/) and accessed at:
```
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json
```

It includes clinical, hormone, imaging, and genetic records mapped for three female patients diagnosed with non-classical 21-hydroxylase deficiency-associated infertility.

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load dataset (schema and metadata)
dataset = mlc.Dataset(croissant_url)

# Access and print dataset metadata
metadata = dataset.metadata
print("Dataset Name:", metadata.name)
print("Description:", metadata.description)

## 2. Data Overview
Review available record sets, fields, and their unique `@id`s.

The FAIR^2 dataset contains several record sets corresponding to clinical, hormone/imaging, and genetic variant tables. We'll list their `@id` values and fields.

In [ ]:
# List all record sets and fields using their @id
record_sets = dataset.metadata.record_set

if not record_sets:
    print("No record sets defined directly in metadata."
          "\nIf the schema defines them elsewhere, try: dataset.record_sets.")

# Try accessing record sets from dataset.record_sets
record_sets = dataset.record_sets

for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']} | Name: {rs.get('name', 'N/A')}")
    fields = rs.get('field', [])
    if fields:
        print("  Fields:")
        for f in fields:
            if isinstance(f, dict):
                print(f"    @id: {f['@id']} | Name: {f.get('name', 'N/A')}")
            else:
                print(f"    @id: {f}")
    print()

# Show an example record from each set (dynamic by @id)
for rs in record_sets:
    rs_id = rs['@id']
    print(f"Example record from RecordSet {rs_id}:")
    recs = list(dataset.records(record_set=rs_id))
    if recs:
        print(recs[0])
    else:
        print("No records found.")
    print()

## 3. Data Extraction
Extract data from each record set using their `@id`s. We'll load the data into pandas DataFrames, using only the record set `@id`s found above.

In [ ]:
# Get the @id for each RecordSet found above
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"No records loaded for {record_set_id}")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nColumns for RecordSet {record_set_id}:")
    print(df.columns.tolist())
    print("Sample data:")
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing: filter records, normalize numeric fields, and group records by categorical fields. All fields and columns are referenced by their `@id`.

Let's choose a numeric field (`age at diagnosis` from the clinical table) and a categorical/group field (`phenotype` from the genetic variants table) as example analyses.

In [ ]:
# Example: Find record set with "age at diagnosis" (clin features)
clinical_rs_id = None
clinical_numeric_field_id = None

for rs in dataset.record_sets:
    fields = rs.get('field', [])
    for f in fields:
        if isinstance(f, dict) and ('age' in f.get('name', '').lower()):
            clinical_rs_id = rs['@id']
            clinical_numeric_field_id = f['@id']
            break
    if clinical_rs_id:
        break

if not clinical_rs_id:
    print("No clinical record set with 'age at diagnosis' found.")
else:
    df_clinical = dataframes[clinical_rs_id]
    # Use the actual column if it differs from @id
    col_candidates = [col for col in df_clinical.columns if ('age' in col.lower())]
    numeric_field_col = col_candidates[0] if col_candidates else clinical_numeric_field_id
    threshold = 25
    filtered_df = df_clinical[df_clinical[numeric_field_col] > threshold]
    print(f"Filtered records with {numeric_field_col} > {threshold}:")
    print(filtered_df.head())
    filtered_df[f"{numeric_field_col}_normalized"] = (filtered_df[numeric_field_col] - filtered_df[numeric_field_col].mean()) / filtered_df[numeric_field_col].std()
    print(f"Normalized {numeric_field_col} for filtered records:")
    print(filtered_df[[numeric_field_col, f"{numeric_field_col}_normalized"]].head())

# Example: Group records in genetic variant table by inferred phenotype
variant_rs_id = None
group_field_id = None
for rs in dataset.record_sets:
    fields = rs.get('field', [])
    for f in fields:
        if isinstance(f, dict) and ('phenotype' in f.get('name', '').lower()):
            variant_rs_id = rs['@id']
            group_field_id = f['@id']
            break
    if variant_rs_id:
        break

if variant_rs_id and group_field_id:
    df_genotype = dataframes.get(variant_rs_id, pd.DataFrame())
    # Use the actual column if it differs from @id
    col_candidates = [col for col in df_genotype.columns if ('phenotype' in col.lower())]
    group_field_col = col_candidates[0] if col_candidates else group_field_id
    if group_field_col in df_genotype:
        grouped_df = df_genotype.groupby(group_field_col).mean(numeric_only=True)
        print(f"Grouped genotype variant data by {group_field_col}:\n")
        print(grouped_df.head())

## 5. Visualization
Visualize distributions, comparisons, or relationships using DataFrames.

Below are some basic plots for clinical age and hormone levels versus phenotype. All references are by record set and field `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize age distribution in clinical table
if clinical_rs_id and numeric_field_col in df_clinical.columns:
    plt.figure(figsize=(6, 4))
    sns.histplot(df_clinical[numeric_field_col], kde=True)
    plt.title(f"Distribution of {numeric_field_col} (RecordSet: {clinical_rs_id})")
    plt.xlabel(numeric_field_col)
    plt.ylabel("Count")
    plt.show()

# Visualize hormone levels by inferred phenotype (if available)
if variant_rs_id and group_field_col in df_genotype.columns:
    hormone_cols = [col for col in df_genotype.columns if 'hormone' in col.lower()]
    if hormone_cols:
        plt.figure(figsize=(8, 4))
        for col in hormone_cols:
            sns.boxplot(x=group_field_col, y=col, data=df_genotype)
            plt.title(f"{col} by {group_field_col}")
            plt.show()

## 6. Conclusion
Through the `mlcroissant` library and Croissant schema, we've loaded clinical, hormone, imaging, and genetic variant data for rare disease infertility cases.

Key findings:
- The dataset comprises three female patients with detailed clinical and genotype data, accessible by Croissant schema `@id`s.
- Numeric fields such as age at diagnosis and hormone levels can be easily filtered and normalized; categorical genotype-phenotype association can be explored and visualized.
- Due to its small cohort, results should be interpreted mainly for academic purposes, not for ML model development.

To explore your own analysis, reference entities by their schema `@id`, as demonstrated above.